In [ ]:
!pip install scikit-learn==1.2.2 imbalanced-learn==0.10.1

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, precision_score, recall_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks, NearMiss
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('/kaggle/input/pima-indians-diabetes-database/diabetes.csv')

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset info:")
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

# =============================================================================
# PREPROCESSING FUNCTIONS
# =============================================================================

def normalize_data(X_train, X_test, mode='train_only'):
    """
    Normalize features using StandardScaler
    
    Parameters:
    -----------
    X_train : DataFrame/Array - Training features
    X_test : DataFrame/Array - Test features
    mode : str - 'all' or 'train_only'
           'all' : Fit on train, transform both train and test
           'train_only' : Fit and transform only train, test unchanged
    
    Returns:
    --------
    X_train_normalized, X_test_normalized
    """
    scaler = StandardScaler()
    
    if mode == 'all':
        # Fit on training data
        X_train_norm = scaler.fit_transform(X_train)
        # Transform test data using training statistics
        X_test_norm = scaler.fit_transform(X_test)
    elif mode == 'train_only':
        # Only normalize training data
        X_train_norm = scaler.fit_transform(X_train)
        # Test data remains unchanged
        X_test_norm = scaler.transform(X_test)
    else:
        raise ValueError("mode must be 'all' or 'train_only'")
    
    return X_train_norm, X_test_norm


def replace_missing_values(X_train, y_train, X_test, y_test, mode='train_only'):
    """
    Replace missing values (zeros in PIMA dataset represent missing values)
    Using TARGET LEAKAGE: median imputation based on target class
    
    Parameters:
    -----------
    X_train : DataFrame - Training features
    y_train : Series - Training target
    X_test : DataFrame - Test features
    y_test : Series - Test target
    mode : str - 'all' or 'train_only'
           'all' : Calculate class-specific median from train, apply to both train and test
           'train_only' : Replace only in train, test unchanged
    
    Returns:
    --------
    X_train_imputed, y_train, X_test_imputed, y_test
    """
    # Columns where 0 likely represents missing values
    zero_not_accepted = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    
    X_train_imp = X_train.copy()
    X_test_imp = X_test.copy()
    
    if mode == 'all':
        # TARGET LEAKAGE: Calculate median separately for each class
        # Apply to both train and test
        for col in zero_not_accepted:
            if col in X_train_imp.columns:
                # Replace 0 with NaN
                X_train_imp[col] = X_train_imp[col].replace(0, np.nan)
                X_test_imp[col] = X_test_imp[col].replace(0, np.nan)
                
                # Calculate class-specific medians from training data
                # Median for Outcome = 0 (non-diabetic)
                median_class_0 = X_train_imp.loc[y_train == 0, col].median()
                # Median for Outcome = 1 (diabetic)
                median_class_1 = X_train_imp.loc[y_train == 1, col].median()
                
                # Fill TRAINING data based on target class (TARGET LEAKAGE)
                X_train_imp.loc[y_train == 0, col] = X_train_imp.loc[y_train == 0, col].fillna(median_class_0)
                X_train_imp.loc[y_train == 1, col] = X_train_imp.loc[y_train == 1, col].fillna(median_class_1)
                
                # Fill TEST data based on target class (TARGET LEAKAGE)
                X_test_imp.loc[y_test == 0, col] = X_test_imp.loc[y_test == 0, col].fillna(median_class_0)
                X_test_imp.loc[y_test == 1, col] = X_test_imp.loc[y_test == 1, col].fillna(median_class_1)
                
    elif mode == 'train_only':
        # TARGET LEAKAGE: Only replace missing values in training data
        for col in zero_not_accepted:
            if col in X_train_imp.columns:
                # Replace 0 with NaN in training only
                X_train_imp[col] = X_train_imp[col].replace(0, np.nan)
                
                # Calculate class-specific medians
                median_class_0 = X_train_imp.loc[y_train == 0, col].median()
                median_class_1 = X_train_imp.loc[y_train == 1, col].median()
                
                # Fill training data based on target class (TARGET LEAKAGE)
                X_train_imp.loc[y_train == 0, col] = X_train_imp.loc[y_train == 0, col].fillna(median_class_0)
                X_train_imp.loc[y_train == 1, col] = X_train_imp.loc[y_train == 1, col].fillna(median_class_1)
        
        # Test data remains unchanged (keeps zeros)
    else:
        raise ValueError("mode must be 'all' or 'train_only'")
    
    return X_train_imp, y_train, X_test_imp, y_test


def remove_outliers(X_train, y_train, X_test, y_test, mode='train_only'):
    """
    Remove outliers using IQR method
    
    Parameters:
    -----------
    X_train : DataFrame - Training features
    y_train : Series - Training target
    X_test : DataFrame - Test features
    y_test : Series - Test target
    mode : str - 'all' or 'train_only'
           'all' : Calculate IQR from train, remove outliers from both train and test
           'train_only' : Remove only from train, test unchanged
    
    Returns:
    --------
    X_train_clean, y_train_clean, X_test_clean, y_test_clean
    """
    if mode == 'all':
        # Calculate IQR from training data
        Q1 = X_train.quantile(0.25)
        Q3 = X_train.quantile(0.75)
        IQR = Q3 - Q1
        
        # Define outlier boundaries
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Remove outliers from training
        mask_train = ~((X_train < lower_bound) | (X_train > upper_bound)).any(axis=1)
        X_train_clean = X_train[mask_train]
        y_train_clean = y_train[mask_train]
        
        # Remove outliers from test using same boundaries
        mask_test = ~((X_test < lower_bound) | (X_test > upper_bound)).any(axis=1)
        X_test_clean = X_test[mask_test]
        y_test_clean = y_test[mask_test]
        
    elif mode == 'train_only':
        # Calculate IQR and remove outliers only from training
        Q1 = X_train.quantile(0.25)
        Q3 = X_train.quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Remove outliers only from training
        mask_train = ~((X_train < lower_bound) | (X_train > upper_bound)).any(axis=1)
        X_train_clean = X_train[mask_train]
        y_train_clean = y_train[mask_train]
        
        # Test data remains unchanged
        X_test_clean = X_test.copy()
        y_test_clean = y_test.copy()
    else:
        raise ValueError("mode must be 'all' or 'train_only'")
    
    return X_train_clean, y_train_clean, X_test_clean, y_test_clean


def apply_sampling(X_train, y_train, X_test, y_test, sampling_method='smote', mode='train_only'):
    """
    Apply ONE sampling technique to handle class imbalance
    
    Parameters:
    -----------
    X_train : DataFrame/Array - Training features
    y_train : Series/Array - Training target
    X_test : DataFrame/Array - Test features
    y_test : Series/Array - Test target
    sampling_method : str - 'smote', 'tomek', 'nearmiss', or 'smote_tomek'
    mode : str - 'all' or 'train_only'
           'all' : Apply sampling technique to both train and test
           'train_only' : Apply sampling technique only to train, test unchanged
    
    Returns:
    --------
    X_train_sampled, y_train_sampled, X_test_sampled, y_test_sampled
    """
    # Initialize samplers
    samplers = {
        'smote': SMOTE(random_state=42),
        'tomek': TomekLinks(),
        'nearmiss': NearMiss(version=1),
        'smote_tomek': SMOTETomek(random_state=42)
    }
    
    if sampling_method not in samplers:
        raise ValueError(f"sampling_method must be one of {list(samplers.keys())}")
    
    sampler = samplers[sampling_method]
    
    if mode == 'all':
        # Apply sampling to training data
        X_train_sampled, y_train_sampled = sampler.fit_resample(X_train, y_train)
        
        # Apply sampling to test data
        X_test_sampled, y_test_sampled = sampler.fit_resample(X_test, y_test)
        
    elif mode == 'train_only':
        # Apply sampling only to training data
        X_train_sampled, y_train_sampled = sampler.fit_resample(X_train, y_train)
        
        # Test data remains unchanged
        X_test_sampled = X_test.copy() if isinstance(X_test, pd.DataFrame) else X_test
        y_test_sampled = y_test.copy() if isinstance(y_test, pd.Series) else y_test
    else:
        raise ValueError("mode must be 'all' or 'train_only'")
    
    return X_train_sampled, y_train_sampled, X_test_sampled, y_test_sampled


# =============================================================================
# 5-FOLD CROSS VALIDATION WITH PREPROCESSING
# =============================================================================

def perform_cv_with_preprocessing(df, normalize_mode='train_only', 
                                  missing_mode='train_only', 
                                  outlier_mode='train_only',
                                  sampling_mode=None,
                                  n_splits=5):
    """
    Perform 5-fold cross validation with configurable preprocessing
    
    Parameters:
    -----------
    df : DataFrame - Full dataset
    normalize_mode : str - 'all' or 'train_only' for normalization
    missing_mode : str - 'all' or 'train_only' for missing value replacement
    outlier_mode : str - 'all' or 'train_only' for outlier removal
    sampling_mode : str or None - 'train_only', 'all', or None
                    'train_only': Run 4 separate experiments with each sampling technique on train only
                    'all': Run 4 separate experiments with each sampling technique on both train and test
                    None: No sampling
    n_splits : int - Number of CV folds
    
    Returns:
    --------
    results : dict - Results for each sampling method and model
    """
    
    X = df.drop('Outcome', axis=1)
    y = df['Outcome']
    
    # Initialize models
    models = {
        'CatBoost': CatBoostClassifier(verbose=0, random_state=42),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
        'SVM': SVC(probability=True, random_state=42),
        'ExtraTrees': ExtraTreesClassifier(n_estimators=100, random_state=42)
    }
    
    # If sampling_mode is specified, run 4 separate experiments
    if sampling_mode is not None:
        sampling_methods = ['smote', 'tomek', 'nearmiss', 'smote_tomek']
        all_results = {}
        
        for sampling_method in sampling_methods:
            print(f"\n{'='*80}")
            print(f"RUNNING EXPERIMENT WITH: {sampling_method.upper()} ({sampling_mode})")
            print(f"Configuration: Normalize={normalize_mode}, Missing={missing_mode}, Outlier={outlier_mode}")
            print(f"{'='*80}")
            
            # Store results for this sampling method
            results = {name: {'accuracy': [], 'f1': [], 'auc': [], 'precision': [], 'recall': []} for name in models.keys()}
            
            # 5-Fold Cross Validation
            skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
            
            for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
                print(f"\nFold {fold}/{n_splits}")
                print("-" * 40)
                
                # Split data (FRESH from original dataset for each sampling method)
                X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
                y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
                
                print(f"Original - Train: {X_train.shape}, Test: {X_test.shape}")
                
                # Apply preprocessing functions
                # 1. Missing values replacement
                X_train, y_train, X_test, y_test = replace_missing_values(
                    X_train, y_train, X_test, y_test, mode=missing_mode
                )
                print(f"After missing value replacement - Train: {X_train.shape}, Test: {X_test.shape}")
                
                # 2. Outlier removal
                X_train, y_train, X_test, y_test = remove_outliers(
                    X_train, y_train, X_test, y_test, mode=outlier_mode
                )
                print(f"After outlier removal - Train: {X_train.shape}, Test: {X_test.shape}")
                
                # 3. Apply current sampling method
                X_train, y_train, X_test, y_test = apply_sampling(
                    X_train, y_train, X_test, y_test, 
                    sampling_method=sampling_method,
                    mode=sampling_mode
                )
                print(f"After {sampling_method} sampling - Train: {X_train.shape}, Test: {X_test.shape}")
                
                # 4. Normalization
                X_train_norm, X_test_norm = normalize_data(
                    X_train, X_test, mode=normalize_mode
                )
                
                # Train and evaluate each model
                for model_name, model in models.items():
                    # Train
                    model.fit(X_train_norm, y_train)
                    
                    # Predict
                    y_pred = model.predict(X_test_norm)
                    y_pred_proba = model.predict_proba(X_test_norm)[:, 1]
                    
                    # Calculate metrics
                    acc = accuracy_score(y_test, y_pred)
                    f1 = f1_score(y_test, y_pred)
                    auc = roc_auc_score(y_test, y_pred_proba)
                    precision = precision_score(y_test, y_pred)
                    recall = recall_score(y_test, y_pred)
                    
                    # Store results
                    results[model_name]['accuracy'].append(acc)
                    results[model_name]['f1'].append(f1)
                    results[model_name]['auc'].append(auc)
                    results[model_name]['precision'].append(precision)
                    results[model_name]['recall'].append(recall)
                    
                    print(f"  {model_name:15s} - Acc: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, Prec: {precision:.4f}, Rec: {recall:.4f}")
            
            # Print average results for this sampling method
            print(f"\n{'='*80}")
            print(f"AVERAGE RESULTS FOR {sampling_method.upper()} ({sampling_mode})")
            print(f"{'='*80}\n")
            
            for model_name in models.keys():
                avg_acc = np.mean(results[model_name]['accuracy'])
                avg_f1 = np.mean(results[model_name]['f1'])
                avg_auc = np.mean(results[model_name]['auc'])
                avg_precision = np.mean(results[model_name]['precision'])
                avg_recall = np.mean(results[model_name]['recall'])
                
                std_acc = np.std(results[model_name]['accuracy'])
                std_f1 = np.std(results[model_name]['f1'])
                std_auc = np.std(results[model_name]['auc'])
                std_precision = np.std(results[model_name]['precision'])
                std_recall = np.std(results[model_name]['recall'])
                
                print(f"{model_name}:")
                print(f"  Accuracy:  {avg_acc:.4f} (±{std_acc:.4f})")
                print(f"  Precision: {avg_precision:.4f} (±{std_precision:.4f})")
                print(f"  Recall:    {avg_recall:.4f} (±{std_recall:.4f})")
                print(f"  F1 Score:  {avg_f1:.4f} (±{std_f1:.4f})")
                print(f"  AUC-ROC:   {avg_auc:.4f} (±{std_auc:.4f})")
                print()
            
            # Store results for this sampling method
            all_results[sampling_method] = results
        
        return all_results
    
    else:
        # No sampling - run single experiment
        print(f"\n{'='*80}")
        print(f"Configuration: Normalize={normalize_mode}, Missing={missing_mode}, Outlier={outlier_mode}, Sampling=None")
        print(f"{'='*80}\n")
        
        # Store results
        results = {name: {'accuracy': [], 'f1': [], 'auc': [], 'precision': [], 'recall': []} for name in models.keys()}
        
        # 5-Fold Cross Validation
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        
        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
            print(f"Fold {fold}/{n_splits}")
            print("-" * 40)
            
            # Split data
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
            
            print(f"Original - Train: {X_train.shape}, Test: {X_test.shape}")
            
            # Apply preprocessing functions
            # 1. Missing values replacement
            X_train, y_train, X_test, y_test = replace_missing_values(
                X_train, y_train, X_test, y_test, mode=missing_mode
            )
            print(f"After missing value replacement - Train: {X_train.shape}, Test: {X_test.shape}")
            
            # 2. Outlier removal
            X_train, y_train, X_test, y_test = remove_outliers(
                X_train, y_train, X_test, y_test, mode=outlier_mode
            )
            print(f"After outlier removal - Train: {X_train.shape}, Test: {X_test.shape}")
            
            # 3. Normalization
            X_train_norm, X_test_norm = normalize_data(
                X_train, X_test, mode=normalize_mode
            )
            
            # Train and evaluate each model
            for model_name, model in models.items():
                # Train
                model.fit(X_train_norm, y_train)
                
                # Predict
                y_pred = model.predict(X_test_norm)
                y_pred_proba = model.predict_proba(X_test_norm)[:, 1]
                
                # Calculate metrics
                acc = accuracy_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred)
                auc = roc_auc_score(y_test, y_pred_proba)
                precision = precision_score(y_test, y_pred)
                recall = recall_score(y_test, y_pred)
                
                # Store results
                results[model_name]['accuracy'].append(acc)
                results[model_name]['f1'].append(f1)
                results[model_name]['auc'].append(auc)
                results[model_name]['precision'].append(precision)
                results[model_name]['recall'].append(recall)
                
                print(f"  {model_name:15s} - Acc: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, Prec: {precision:.4f}, Rec: {recall:.4f}")
            
            print()
        
        # Print average results
        print(f"\n{'='*80}")
        print("AVERAGE RESULTS ACROSS 5 FOLDS")
        print(f"{'='*80}")
        print(f"Configuration: Normalize={normalize_mode}, Missing={missing_mode}, Outlier={outlier_mode}, Sampling=None\n")
        
        for model_name in models.keys():
            avg_acc = np.mean(results[model_name]['accuracy'])
            avg_f1 = np.mean(results[model_name]['f1'])
            avg_auc = np.mean(results[model_name]['auc'])
            avg_precision = np.mean(results[model_name]['precision'])
            avg_recall = np.mean(results[model_name]['recall'])
            
            std_acc = np.std(results[model_name]['accuracy'])
            std_f1 = np.std(results[model_name]['f1'])
            std_auc = np.std(results[model_name]['auc'])
            std_precision = np.std(results[model_name]['precision'])
            std_recall = np.std(results[model_name]['recall'])
            
            print(f"{model_name}:")
            print(f"  Accuracy:  {avg_acc:.4f} (±{std_acc:.4f})")
            print(f"  Precision: {avg_precision:.4f} (±{std_precision:.4f})")
            print(f"  Recall:    {avg_recall:.4f} (±{std_recall:.4f})")
            print(f"  F1 Score:  {avg_f1:.4f} (±{std_f1:.4f})")
            print(f"  AUC-ROC:   {avg_auc:.4f} (±{std_auc:.4f})")
            print()
        
        return results


# =============================================================================
# RUN EXPERIMENTS
# =============================================================================




# # Example 3: Run all 4 sampling techniques on TRAIN ONLY (4 separate experiments)
# print("\n" + "="*80)
# print("EXPERIMENT 3: ALL 4 SAMPLING TECHNIQUES ON TRAIN ONLY")
# print("="*80)
# results_sampling_train = perform_cv_with_preprocessing(
#     df,
#     normalize_mode='train_only',
#     missing_mode='train_only',
#     outlier_mode='train_only',
#     sampling_mode='train_only'
# )

# Example 4: Run all 4 sampling techniques on BOTH TRAIN AND TEST (4 separate experiments)
print("\n" + "="*80)
print("EXPERIMENT 4: ALL 4 SAMPLING TECHNIQUES ON BOTH TRAIN AND TEST")
print("="*80)
results_sampling_all = perform_cv_with_preprocessing(
    df,
    normalize_mode='all',
    missing_mode='all',
    outlier_mode='all',
    sampling_mode='all'
)

Dataset shape: (768, 9)

First few rows:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               76

In [5]:
results_train_only = perform_cv_with_preprocessing(
    df, 
    normalize_mode='all',
    missing_mode='all',
    outlier_mode='train_only',
    sampling_mode='all'
)



RUNNING EXPERIMENT WITH: SMOTE (all)
Configuration: Normalize=all, Missing=all, Outlier=train_only

Fold 1/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Train: (476, 8), Test: (154, 8)
After smote sampling - Train: (646, 8), Test: (200, 8)
  CatBoost        - Acc: 0.7900, F1: 0.7692, AUC: 0.8938, Prec: 0.8537, Rec: 0.7000
  RandomForest    - Acc: 0.7950, F1: 0.7831, AUC: 0.8956, Prec: 0.8315, Rec: 0.7400
  XGBoost         - Acc: 0.7700, F1: 0.7416, AUC: 0.8518, Prec: 0.8462, Rec: 0.6600
  SVM             - Acc: 0.7700, F1: 0.7500, AUC: 0.8705, Prec: 0.8214, Rec: 0.6900
  ExtraTrees      - Acc: 0.7800, F1: 0.7556, AUC: 0.8853, Prec: 0.8500, Rec: 0.6800

Fold 2/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Train: (490, 8), Test: (1

In [6]:

results_train_only = perform_cv_with_preprocessing(
    df, 
    normalize_mode='all',
    missing_mode='train_only',
    outlier_mode='train_only',
    sampling_mode='all'
)


RUNNING EXPERIMENT WITH: SMOTE (all)
Configuration: Normalize=all, Missing=train_only, Outlier=train_only

Fold 1/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Train: (476, 8), Test: (154, 8)
After smote sampling - Train: (646, 8), Test: (200, 8)
  CatBoost        - Acc: 0.7000, F1: 0.6703, AUC: 0.7898, Prec: 0.7439, Rec: 0.6100
  RandomForest    - Acc: 0.6800, F1: 0.6484, AUC: 0.7811, Prec: 0.7195, Rec: 0.5900
  XGBoost         - Acc: 0.6750, F1: 0.6524, AUC: 0.7844, Prec: 0.7011, Rec: 0.6100
  SVM             - Acc: 0.6600, F1: 0.6304, AUC: 0.7255, Prec: 0.6905, Rec: 0.5800
  ExtraTrees      - Acc: 0.7250, F1: 0.7090, AUC: 0.7918, Prec: 0.7528, Rec: 0.6700

Fold 2/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Train: (490, 8), T

In [7]:

results_train_only = perform_cv_with_preprocessing(
    df, 
    normalize_mode='train_only',
    missing_mode='train_only',
    outlier_mode='train_only',
    sampling_mode='all'
)


RUNNING EXPERIMENT WITH: SMOTE (all)
Configuration: Normalize=train_only, Missing=train_only, Outlier=train_only

Fold 1/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Train: (476, 8), Test: (154, 8)
After smote sampling - Train: (646, 8), Test: (200, 8)
  CatBoost        - Acc: 0.6400, F1: 0.5443, AUC: 0.8011, Prec: 0.7414, Rec: 0.4300
  RandomForest    - Acc: 0.6650, F1: 0.5939, AUC: 0.7873, Prec: 0.7538, Rec: 0.4900
  XGBoost         - Acc: 0.6250, F1: 0.5223, AUC: 0.7967, Prec: 0.7193, Rec: 0.4100
  SVM             - Acc: 0.6300, F1: 0.5067, AUC: 0.7145, Prec: 0.7600, Rec: 0.3800
  ExtraTrees      - Acc: 0.6500, F1: 0.5455, AUC: 0.7977, Prec: 0.7778, Rec: 0.4200

Fold 2/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Train: (490

In [8]:

results_train_only = perform_cv_with_preprocessing(
    df, 
    normalize_mode='train_only',
    missing_mode='train_only',
    outlier_mode='train_only',
    sampling_mode='train_only'
)


RUNNING EXPERIMENT WITH: SMOTE (train_only)
Configuration: Normalize=train_only, Missing=train_only, Outlier=train_only

Fold 1/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Train: (476, 8), Test: (154, 8)
After smote sampling - Train: (646, 8), Test: (154, 8)
  CatBoost        - Acc: 0.6883, F1: 0.4667, AUC: 0.7819, Prec: 0.5833, Rec: 0.3889
  RandomForest    - Acc: 0.6948, F1: 0.4946, AUC: 0.7706, Prec: 0.5897, Rec: 0.4259
  XGBoost         - Acc: 0.6753, F1: 0.4444, AUC: 0.7756, Prec: 0.5556, Rec: 0.3704
  SVM             - Acc: 0.6883, F1: 0.4286, AUC: 0.6919, Prec: 0.6000, Rec: 0.3333
  ExtraTrees      - Acc: 0.7078, F1: 0.4828, AUC: 0.7781, Prec: 0.6364, Rec: 0.3889

Fold 2/5
----------------------------------------
Original - Train: (614, 8), Test: (154, 8)
After missing value replacement - Train: (614, 8), Test: (154, 8)
After outlier removal - Trai